### Common Set Operations

Let's look at some of the more basic and common operations with sets:
* size
* membership testing
* adding elements
* removing elements

#### Size

The size of a set (it's cardinality), is given by the `len()` function - the same one we use for sequences, iterables, dictionaries, etc.

In [1]:
s = {1, 2, 3}
len(s)

3

#### Membership Testing

This is also very easy:

In [2]:
s = {1, 2, 3}

In [3]:
1 in s

True

In [4]:
10 in s

False

In [5]:
1 not in s

False

In [6]:
10 not in s

True

But let's go a little further and consider how membership testing works with sets. As I mentioned in earlier lectures, sets are hash tables, and membership testing is **extremely** efficient for sets, since it's simply a **hash table lookup** - as opposed to scanning a list for example, until we find the requested element (or not).

Let's do some quick timings to verify this, as well as compare lookup speeds for sets and dictionaries as well (which are also, after all, hash tables).

In [7]:
from timeit import timeit

In [8]:
n = 100_000
s = {i for i in range(n)}
l = [i for i in range(n)]
d = {i: None for i in range(n)}

Let's time how long it takes to find if `9` is in the object - which would be the tenth element only of the list and the dictionary (keys), and who knows for the set:

In [ ]:
number = 1_000_000
search = 9
# '9 in l' - statement to use in timeit function
# globals=globals - gives access to s, l, d
# number=number - run this number of times
t_list = timeit(f'{search} in l', globals=globals(), number=number)
t_set = timeit(f'{search} in s', globals=globals(), number=number)
t_dict = timeit(f'{search} in d', globals=globals(), number=number)
print('list:', t_list)
print('set:', t_set)
print('dict:', t_dict)

list: 0.07541930000297725
set: 0.01742489999742247
dict: 0.019653199997264892


The story changes even more if we test for example the last element of the list.
I'm definitely not to run the tests `1_000_000` times - not unless we want to make this video reaaaaaaly long!

In [15]:
number = 3_000
search = 99_999
t_list = timeit(f'{search} in l', globals=globals(), number=number)
t_set = timeit(f'{search} in s', globals=globals(), number=number)
t_dict = timeit(f'{search} in d', globals=globals(), number=number)
print('list:', t_list)
print('set:', t_set)
print('dict:', t_dict)

list: 1.8368648000177927
set: 6.960000609979033e-05
dict: 7.310000364668667e-05


Set is **26,000× faster** than list here. Dict is almost identical to set.

#### Why

|Structure|How `x in ...` works|Complexity|
|---|---|---|
|`list`|Scans every element from index 0 until it finds a match or exhausts the list|$O(n)$|
|`set`|Computes `hash(x)`, jumps directly to that bucket|$O(1)$ average|
|`dict`|Same hash table — checks keys only|$O(1)$ average|

The list time scales with the size of the list. If you doubled the list, the list time would roughly double; the set and dict times would stay the same.

#### Why set ≈ dict

They're both backed by the same hash table implementation in CPython. A `set` is essentially a `dict` with keys but no values. The lookup path is the same code.

#### The practical rule

Any time you write `x in some_list` inside a loop — or anywhere the list has more than a handful of elements — convert it to a set first:

```python
# ❌ O(n) per lookup — hurts when called repeatedly
allowed_roles = ["admin", "editor", "viewer"]
if user.role in allowed_roles: ...

# ✅ O(1) per lookup
ALLOWED_ROLES = {"admin", "editor", "viewer"}
if user.role in ALLOWED_ROLES: ...
```

The conversion `set(some_list)` is $O(n)$ once. Every lookup after that is $O(1)$.

The situation for `not in` is the same:

In [16]:
number = 3_000
search = -1
t_list = timeit(f'{search} not in l', globals=globals(), number=number)
t_set = timeit(f'{search} not in s', globals=globals(), number=number)
t_dict = timeit(f'{search} not in d', globals=globals(), number=number)
print('list:', t_list)
print('set:', t_set)
print('dict:', t_dict)

list: 1.788628600013908
set: 5.110001075081527e-05
dict: 5.110001075081527e-05


But this efficiency does come at the cost of memory:

In [18]:
print('dict:', d.__sizeof__())
print('set: ', s.__sizeof__())
print('list:', l.__sizeof__())

dict: 5242944
set:  4194504
list: 800968


The dict is the haviest since it contains key and value pairs. A bit lighter is set, only containing keys, but is still havier than lists, and the lightest is list

Even for empty objects:

In [1]:
s = set()
d = dict()
l = list()

In [2]:
print('set: ', s.__sizeof__())
print('dict:', d.__sizeof__())
print('list:', l.__sizeof__())

set:  200
dict: 48
list: 40


#### `__sizeof__()` only counts inside the object boundary

List and dict store their backing array as a **separate heap allocation** — `__sizeof__()` does not follow that pointer.

```shell
list  → 40 bytes (just the PyListObject shell)
          + external array when populated → not counted

dict  → 48 bytes (just the PyDictObject shell with 4 fields)
          + PyDictKeysObject on the heap → not counted
```
So the comparison you're looking at is not apples-to-apples. The set looks heavy because its minimum table is _inside_ the object. The dict looks light because its table is _outside_.

#### The honest picture with `sys.getsizeof`

In [5]:
import sys

s = set()
d = dict()
l = list()

print(sys.getsizeof(s))   # 216   (200 + GC header — still all inline)
print(sys.getsizeof(d))   # 64    (48 + GC header — table NOT counted)
print(sys.getsizeof(l))   # 56    (40 + GC header — array NOT counted)

216
64
56


A freshly created empty dict's `ma_keys` object is another ~232 bytes on the heap that `sys.getsizeof` silently ignores.

### So which is actually heavier at scale?

At small sizes, set's embedded table makes it appear heavier. Once either structure grows past 8 entries (the inline table fills and set allocates an external table too), the playing field levels out — both become dominated by their external hash table allocations and the embedded 128 bytes become a rounding error.

**The takeaway:** `__sizeof__()` is an unreliable proxy for "how much memory does this use?" because it doesn't follow pointers. `tracemalloc` or `memory_profiler` give you the real picture.

And adding just one element to each object:

In [3]:
s.add(10)
d[10] =None
l.append(10)

In [4]:
print(d.__sizeof__())
print(s.__sizeof__())
print(l.__sizeof__())

208
200
72


If you're wondering why the dictionary and set size did not increase, remember when we covered hash tables - there is some overallocation that takes place so we don't incure the cost of resizing every time we had an element. In fact, lists do the same as well - they over-allocate to reduce the resizing cost. I'll come back to that in a minute.

#### Adding Elements

When we have an existing set, we can always add elements to it. Of course *where* it gets "inserted" is unknown. So Python does not call it `append` or `insert` which would connotate ordering of some kind - instead it just calls it `add`:

In [12]:
s = {30, 20, 10}

In [13]:
s.add(15)

In [14]:
s

{10, 15, 20, 30}

Don't be fooled by the apparent ordering of the elements here. This is the same as with dictionaries - Jupyter tries to represent things nicely for us, but underneath the scenes:

In [15]:
print(s)

{10, 20, 30, 15}


In [16]:
s.add(-1)
print(s)

{10, 15, 20, 30, -1}


And the order just changed again! :-)

What's interesting about the `add()` method, is that if we try to add an element that already exists, Python will simply ignore it:

In [17]:
s

{-1, 10, 15, 20, 30}

In [18]:
s.add(15)

In [19]:
s

{-1, 10, 15, 20, 30}

### Resizing

Now that we know how to add an element to a set, let's go back and see how  the set, dictionary and list resize as we add more elements to them.
We should expect the list to be more efficient from a memory standpoint:

In [31]:
l = list()
s = set()
d = dict()

print('#', 'dict', 'set', 'list')
for i in range(50):
    print(i, d.__sizeof__(), s.__sizeof__(), l.__sizeof__())
    l.append(i)
    s.add(i)
    d[i] = None

# dict set list
0 48 200 40
1 208 200 72
2 208 200 72
3 208 200 72
4 208 200 72
5 208 712 104
6 336 712 104
7 336 712 104
8 336 712 104
9 336 712 168
10 336 712 168
11 616 712 168
12 616 712 168
13 616 712 168
14 616 712 168
15 616 712 168
16 616 712 168
17 616 712 232
18 616 712 232
19 616 2248 232
20 616 2248 232
21 616 2248 232
22 1152 2248 232
23 1152 2248 232
24 1152 2248 232
25 1152 2248 296
26 1152 2248 296
27 1152 2248 296
28 1152 2248 296
29 1152 2248 296
30 1152 2248 296
31 1152 2248 296
32 1152 2248 296
33 1152 2248 360
34 1152 2248 360
35 1152 2248 360
36 1152 2248 360
37 1152 2248 360
38 1152 2248 360
39 1152 2248 360
40 1152 2248 360
41 1152 2248 456
42 1152 2248 456
43 2248 2248 456
44 2248 2248 456
45 2248 2248 456
46 2248 2248 456
47 2248 2248 456
48 2248 2248 456
49 2248 2248 456


As you can see, the memory costs for a set or a dict are definitely higher than for a list. You can also see from this how it looks like CPython implements different resizing strategies for sets, dicts and lists.
The strategy by the way has nothing to do with the size of the elements we put in those objects:

In [33]:
l = list()
s = set()
d = dict()

print('#', 'dict', 'set', 'list')
for i in range(50):
    print(i, d.__sizeof__(), s.__sizeof__(), l.__sizeof__())
    l.append(i**1000)
    s.add(i*1000)
    d[i*1000] = None

# dict set list
0 48 200 40
1 208 200 72
2 208 200 72
3 208 200 72
4 208 200 72
5 208 712 104
6 336 712 104
7 336 712 104
8 336 712 104
9 336 712 168
10 336 712 168
11 616 712 168
12 616 712 168
13 616 712 168
14 616 712 168
15 616 712 168
16 616 712 168
17 616 712 232
18 616 712 232
19 616 2248 232
20 616 2248 232
21 616 2248 232
22 1152 2248 232
23 1152 2248 232
24 1152 2248 232
25 1152 2248 296
26 1152 2248 296
27 1152 2248 296
28 1152 2248 296
29 1152 2248 296
30 1152 2248 296
31 1152 2248 296
32 1152 2248 296
33 1152 2248 360
34 1152 2248 360
35 1152 2248 360
36 1152 2248 360
37 1152 2248 360
38 1152 2248 360
39 1152 2248 360
40 1152 2248 360
41 1152 2248 456
42 1152 2248 456
43 2248 2248 456
44 2248 2248 456
45 2248 2248 456
46 2248 2248 456
47 2248 2248 456
48 2248 2248 456
49 2248 2248 456


As you can see the memory cost of the objects themselves did not change, nor did the sizing strategy (remember that all those objects contain pointers to the data, not the data itself - and a pointer to an object, no matter the size of that object, is the same).
So be careful using `__sizeof__` - it's often only part of the story.

#### Removing Elements

Now let's see how we can remove elements from a set.

Just as with dictionaries, we may be trying to remove an item that does not exist in the set. Depending on whether we want to silently ignore deletion of non-existent elements we can use one of two techniques:

In [20]:
s = {1, 2, 3}

In [21]:
s.remove(1)

In [22]:
s

{2, 3}

In [23]:
s.remove(10)

KeyError: 10

As you can see, we get an exception.

If we don't want the exception we can do it this way:

In [24]:
s.discard(10)

In [25]:
s

{2, 3}

We can also remove (and return) an **arbitrary** element from the set:

In [26]:
s = set('python')

In [27]:
s

{'h', 'n', 'o', 'p', 't', 'y'}

In [28]:
s.pop()

'o'

Note that we **do not know** ahead of time what element will get popped.

Also, popping an empty set will result in a `KeyError` exception:

In [29]:
s = set()
s.pop()

KeyError: 'pop from an empty set'

Something like that might be handy to handle all the elements of a set one at a time without caring for the order in which elements are removed from the set - not that you can, anyway - sets are not ordered!
But this way you can get at the elements of a set without knowing the content of the set (since you need to know the element you are removing with `remove` and `discard`.)

Finally, you can empty out a set by calling the `clear` method:

In [30]:
s = {1, 2, 3}
s.clear()
s

set()